#### Cellene under er hentet fra: https://github.com/HVL-ML/DAT255/blob/main/notebooks/15_gradio_and_streamlit.ipynb

# Gradio (and Streamlit) deployment

For deploying an ML-based app there are various approaches, but [Gradio](gradio.app) and [Streamlit](https://streamlit.io/) are both quick and convenient ways to do so. Typically we would implement this in a plain python (`.py`) file rather than a `.ipynb` notebook, but Gradio works here too, so let's try that first. Streamlit needs to be run directly in python, but the code is given below, so you can try out that too.

In this example we set up an image classifier, where the user can upload an image and get the top 5 class predictions in return.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from PIL import Image

## Set up a pretrained model

Let's download and set up a `MobileNetV2` model, trained on the 1000 classes in the ImageNet dataset. You can change this to anything you like. Remeber, however, to also use the appropriate preprocessing function.

In [2]:
model = MobileNetV2(weights="imagenet")

def classify_image(img: Image.Image):

    # Resize to the input image to what MobileNet expects
    img = img.resize((224, 224))
    arr = np.array(img)

    # Check the color channels, so we can take both grayscale, RGB, and RGBA images as input.
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]

    # Add batch dim, and preprocess
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr.astype("float32"))

    # Predict!
    preds = model.predict(arr, verbose=0)
    top5 = decode_predictions(preds, top=5)[0]  # [(id, label, prob), ...]

    return {label: float(prob) for (_, label, prob) in top5}



In [28]:
def classify_image_twice(img: Image.Image):
    predictions = [{},{},{},{}]
    predictions[0] = classify_image(img)
    predictions[1] = classify_image(img)
    predictions[2] = classify_image(img)
    predictions[3] = classify_image(img)
    return predictions

## Set up the Gradio interface

Check the [documentation](https://www.gradio.app/docs) for the various things we can add here.

In [30]:
import gradio as gr

# Example images
examples = [
    "https://cdn.britannica.com/79/232779-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg",
    "https://cdn.britannica.com/41/126641-050-E1CA0E61/cat-suns-hill-Parthenon-Athens-Greece-Acropolis.jpg",
]

with gr.Blocks(title="Visual vehicle recognition in varying lighting conditions") as demo:
    gr.Markdown("## Visual vehicle recognition in varying lighting conditions")
    gr.Markdown(
        "Upload an image or click an example below. "
    )

    with gr.Row():
        image_input = gr.Image(type="pil", label="Input Image")
        label_output = [{},{},{},{}]
        label_output[0] = gr.Label(num_top_classes=1, label="Bilmerke")
        label_output[1] = gr.Label(num_top_classes=3, label="Bilmodell")
        label_output[2] = gr.Label(num_top_classes=3, label="Årsperiodemodell")
        label_output[3] = gr.Label(num_top_classes=3, label="Farge")

    classify_btn = gr.Button("Classify", variant="primary")
    classify_btn.click(fn=classify_image_twice, inputs=image_input, outputs=label_output)

    gr.Examples(
        examples=examples,
        inputs=image_input,
        outputs=label_output,
        fn=classify_image_twice,
        cache_examples=True
    )

Now we can run it:

In [31]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
Caching examples at: 'c:\Users\johan\Documents\School\Dat191\visual-vehicle-recognition-varying-lighting-conditions\gradio\.gradio\cached_examples\106'
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\johan\Documents\School\ML\venvs\tf220-cpu\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johan\Documents\School\ML\venvs\tf220-cpu\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johan\Documents\School\ML\venvs\tf220-cpu\Lib\site-packages\gradio\blocks.py", line 2152, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johan\Documents\School\ML\venvs\tf220-cpu\Lib\site-packages\gradio\blocks.py", line 1629, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\johan\Documents\School\ML\venvs\tf220